---
title: "Understanding the repo CLI `wt`"
date: "2026-09-30"
categories: ["meta", "dev"]
toc: true
---

Watchtower keeps teaching material, operational code, and historical source in one repository. That creates a navigation problem: a notebook's existence does not say whether it is current, published, or relevant to the chapter an agent is editing. The active catalog and the `wt` CLI answer those questions in stages.

This is a guide to the current implementation. It traces a command from the catalog to a notebook cell, explains why a course has YAML, a home notebook, and sometimes an overview, and maps the tests to the behavior they protect. The recurring example is an agent asked to revise one course chapter: how does it find the right source and gather enough context without reading an entire course?


## Repository map: content and machinery

The active registry is `knowledge/catalog.yaml`. It names logical works: posts, courses, chapters, portfolio pieces, and projects. Active post and course notebooks live under `nb/posts/` and `nb/courses/`; a course occupies its own directory. `nb/photos/photos.ipynb` is a separate personal page. Older posts, courses, portfolio pieces, and projects live under `archive/2026-09-30/`. Archive content remains in Git for reuse, but normal active discovery and site navigation exclude it.

The machinery has a different role. `src/watchtower/cli.py` defines the Typer commands. `knowledge.py` owns the catalog, course contract, validation, and site synchronization. `inspect.py` builds maps, lists, search results, and notebook resolution. `notebook.py` reads and changes cells; `execute.py` runs them; `scaffold.py` and `convert.py` create or import notebooks. `scripts/` contains Make-driven site, résumé, PDF, and project tasks. `assets/resume.yaml` is the résumé source. None of these operational files needs an individual knowledge record.

| Question | Source of truth |
|:--|:--|
| Is this a current logical work? | `knowledge/catalog.yaml` |
| What does a course intend, and what is done? | Its `course.yaml` |
| What should a learner read first? | The course's `index.ipynb` |
| How is a subject taught? | The relevant post or chapter notebook |
| How is a notebook rendered? | Quarto configuration plus stored notebook outputs |
| What did the earlier repository contain? | The explicit archive |

This separation also explains why an image, a test file, or a project source file is not registered as its own artifact. Those files belong to a logical work; registering each would make navigation a file inventory instead of a knowledge map.


## The catalog is an index with publication state

A record uses a stable ID rather than a title as its handle. A chapter record, for example, can look like this:

```yaml
- id: course/signals/02-filtering
  kind: chapter
  title: Filtering
  path: nb/courses/signals/02-filtering.ipynb
  visibility: public
  lifecycle: draft
  relations: []
  parent: course/signals
```

`kind` is one of post, course, chapter, portfolio, or project. `path` points to an existing relative source; a course path is the *directory*, while a chapter path is its notebook. `parent` ties a chapter to a registered course. `relations` holds IDs of related works; validation checks that referenced IDs exist. `visibility` is `public` or `private`. `lifecycle` is `planned`, `draft`, or `published`. These are independent: a private published record remains visible to active `wt` discovery but does not enter the public site.

`wt map`, `wt ls`, and default `wt find` derive active results from this registry. `wt map` groups a course into its home, optional overview, and chapter paths. `wt ls courses` prints the course directory, home, and registered chapter paths. For a course, `wt find` searches `course.yaml` and the home; each registered chapter is searched through its own record. Notebook hits report `[cell N]`; text-file hits report line numbers. Search is case-insensitive.

The catalog controls **discovery**, not every possible direct read. `wt cat` and other notebook operations first resolve a registered ID or path, but an explicit existing notebook path and a unique bare stem can still resolve outside the catalog. This is useful while registering new work and for reading an archived notebook by its full path. It also means that a successful `wt cat orphan` does not certify that `orphan` is active. `wt validate` is the check for unregistered files in active content directories.

Site publication is a second filter. `wt sync-site` adds only records with both `visibility: public` and `lifecycle: published` to Quarto's render list and navigation. The archive is explicitly excluded. The catalog is therefore the common input to active discovery and publication, while the two views apply different filters.


## What course YAML says, and what it cannot say

Each registered course has a `course.yaml` with an ID, purpose, audience, and two separate summaries:

```yaml
id: course/signals
purpose: Help readers reason about sampled signals.
audience: Readers comfortable with Python and basic algebra.
planned:
  summary: Build a filter after introducing sampling and noise.
actualized:
  summary: The sampling lesson and its exercises have been checked.
```

`planned` describes the intended path. `actualized` is an explicit account of completed, checked work. The two fields cannot be collapsed safely: a chapter stub, file, or sidebar entry is not evidence that the lesson works. The implementation does not infer completion from files or test results. The author updates actualized text after checking the material. Conversely, `wt validate` checks that both sections and their string summaries exist; it cannot decide whether the educational claim is true.

`wt render-context` turns the YAML into a short `_course-context.md` in the course directory. The home notebook contains `{{< include _course-context.md >}}`, so readers see the same purpose, audience, planned path, and actualized account that `wt context` returns to an agent. When YAML changes, the include becomes stale until regenerated; validation detects a stale generated include. `make knowledge` regenerates it, validates the catalog, and synchronizes site paths before rendering.

The home, `index.ipynb`, is the learner-facing course README: it explains prerequisites, route, navigation, and how to work through the lessons. An optional `00-overview.ipynb` develops the whole-course technical design. Chapters provide local explanation, examples, and exercises. The repository `README.md` explains Watchtower itself; it is not a substitute for any course home. YAML stays concise because it must be useful to both an agent planning a change and a reader deciding where to start. Teaching details belong in the notebooks.


## The context hierarchy in an actual task

Suppose the target is `course/signals/02-filtering`. Run `.venv/bin/wt context course/signals/02-filtering`. The JSON result has three parts: `artifact` is the catalog record; `course` is the parsed course YAML; `reading_paths` labels the course home, optional overview, and target notebook. It does **not** load every chapter's cells. The target appears in `reading_paths` even when there is no overview.

```text
catalog record + course.yaml
             ↓
index.ipynb: learner route and prerequisites
             ↓
00-overview.ipynb, if present: whole-course technical model
             ↓
02-filtering.ipynb: the requested work
             ↓
adjacent chapter openings, only if a handoff matters
```

Read the home when revising the course promise, prerequisites, navigation, or a learner's path. Read the overview when terminology, system design, or cross-chapter invariants could change. A small typo in the target chapter may need only the YAML contract and that chapter; a changed exercise sequence may require the preceding and following chapter introductions. Read those neighbors to answer a concrete dependency question, not as an automatic sweep of the course.

There is a second, unrelated meaning of context in the CLI. `wt cat <notebook> --index 4 --context 2` prints cells 2 through 6 of **one notebook**, marking the surrounding cells as context. It does not consult `course.yaml` or sibling chapters. The distinction matters because a cell edit needs nearby source for coherence, while course-level work needs the course contract and reading order.

For historical research, opt in with `wt map --archive`, `wt ls courses --archive`, or `wt find "filter" --archive`. An archive hit is a reference to prior source, not a claim that the active course contains or endorses it.


## Following a command through the implementation

`cli.py` is deliberately thin. Typer parses a command and options, then imports the needed module inside that command. `wt map` calls `inspect.repo_map_json()`; `wt context` calls `knowledge.context_json()`; `wt cat` calls `notebook.cat_notebook()`. This keeps a navigation command from loading notebook execution dependencies. The public console scripts `wt` and `watchtower` both enter `watchtower.cli:main`.

A chapter read follows this path:

```text
wt cat course/signals/02-filtering --index 3 --context 1
  → cli.cat parses options
  → notebook.cat_notebook resolves the name
  → inspect.resolve_ipynb consults catalog IDs, then path forms
  → nbformat reads the notebook
  → notebook renders selected cells as Markdown-like text
```

`resolve_ipynb` supports an artifact ID, a relative path with or without `.ipynb`, tier-prefixed paths, and a bare stem when it identifies a notebook. A course ID resolves to its home `index.ipynb`. `wt cat` can select a zero-based index, a half-open range such as `2:5`, a Jupyter tag, or a leading Quarto `#| label:`. A tag may match several cells. The default source limit is 4,096 characters per cell; `--offset` and `--limit` page through long sources, and `--limit 0` removes the read limit. `--with-outputs` adds stored text and error output while summarizing image payloads instead of dumping base64.

The cell layer writes through `nbformat`, not by string editing notebook JSON. `edit-cell` changes one cell's source while retaining its outputs and metadata; `append-cell`, `insert-cell`, `remove-cell`, `tag`, and `clear-outputs` handle other changes. A write uses one locator; unique matches are required for editing and tagging, while removal can intentionally delete all matches of a tag. Source writes are capped at 20,000 characters per cell. Indices shift after insertion or removal, so re-read indices before another positional edit.


## Creation, execution, and publication are separate steps

A typical path from a new source to a rendered page is:

```text
wt new post example --title "Example"    # create and register a draft
wt validate                              # check catalog and active paths
# write notebook cells and inspect them with wt cat
wt diff example                          # compare with a Git base when tracked
# set lifecycle: published when ready
make knowledge                           # include, validate, sync site
make resume                              # rebuild home and posts listing
quarto render                            # inspect the reader-facing result
```

`wt new course <slug> <title>` creates `course.yaml`, `index.ipynb`, a first chapter stub, a generated include, catalog records, and a course sidebar. `wt new chapter` adds a registered chapter and sidebar entry; `wt new section` changes sidebar grouping. `wt import` copies an external notebook into posts or into an existing course, preserves stored outputs, supplies a `python3` kernelspec if needed, and removes a duplicate leading H1 only when it repeats frontmatter title. It then registers the new artifact. `wt register` handles existing portfolio or project work created by other means.

For code cells, `wt run` is the explicit execution path. It uses the notebook's kernelspec, falling back to `python3`, and stores outputs in the notebook. A cell error is stored inline; the CLI exits 1 when any executed cell errors. With `--index N`, it executes the prefix through N in a fresh kernel so imports and variables are available, but writes back only N's output. `wt output` prints stored text and errors and saves image outputs for inspection. Quarto has `execute.enabled: false`: rendering reads those stored outputs and does not rerun code.

`wt validate` checks record shape, unique IDs and paths, valid kinds, visibility and lifecycle, file existence, chapter parents, relation targets, course YAML identity and summaries, home and include markers, generated include freshness, publication constraints, and unregistered active files. It is a structural gate. `make knowledge` runs generation before validation so routine YAML changes can be repaired as part of the build.


## Which tests protect which behavior

The tests use temporary repository fixtures, including an empty catalog and a small `_quarto.yml`. That makes a test's active world explicit and prevents the real archive or current courses from affecting assertions. The suite mixes command-boundary tests, module tests, and actual Jupyter kernel execution.

| Behavior | Command-boundary evidence | Focused evidence |
|:--|:--|:--|
| Catalog map, list, search, and ID resolution | `test_course_cli_context_navigation_and_notebook_id` checks map grouping, course listing, a notebook search hit, ID and path context, and reading/editing by ID. | `test_inspect.py` exercises listing, resolution, and search; `test_knowledge.py` proves an unregistered notebook is absent from active search. |
| Course YAML and generated include | The same CLI test edits YAML, regenerates the include, validates, and checks planned and actualized values in context. | `test_course_contract_is_shared_by_context_and_render` checks the stale-include error, regeneration, and the resulting context and include. |
| Registration, site filter, and archive boundary | `test_cli_register_validate_and_site_visibility` checks public versus private publication and an orphan; `test_cli_scaffold_import_and_archive_are_separate` checks creation/import registration and explicit archive search. | `test_knowledge.py` checks public published render paths, draft/private exclusion, and archive-only search. |
| Cell reads and mutations | The course CLI test appends and edits by catalog ID; `test_cli.py` checks tags and exit behavior. | `test_notebook.py` covers indices, ranges, tags, labels, edits, insertion, removal, limits, and diffs; `test_outputs.py` covers output extraction. |
| Execution and repository tasks | `test_cli.py` checks `wt run` exit codes, kernel hints, and the boundary between `wt` and Make commands. | `test_execute.py` runs real kernels to check outputs, errors, and prefix state; `test_tasks.py` checks Make ordering and preview port handling. |

The division is deliberate. A helper test can prove that `inspect.find_in_src` ignores an orphan; a command test proves that `wt find` reaches that behavior. A test that checks `knowledge.context` alone would miss a broken Typer argument or JSON response, so the command test checks those too. Conversely, broad CLI tests do not exercise every cell locator and failure case, so `test_notebook.py` handles them at the cell layer.


## A test trace and its limits

The first catalog CLI test constructs a `signals` course in an isolated directory. It adds a section and a second chapter, writes different text into `planned.summary` and `actualized.summary`, then runs `render-context` and `validate`. It asserts that `map` lists the home and chapters, `find` points to the stub's cell 0, and `context` returns the two YAML summaries and home path. Finally it reads, appends, and edits the chapter using its catalog ID. That one scenario catches a broken connection among scaffolding, the registry, course context, discovery, and cell operations.

The registration test adds public and private portfolio pages, runs `sync-site`, and inspects Quarto's render list. It then creates an unregistered page and expects `validate` to exit 1 with the orphan path. The archive test creates a historical notebook and checks that default `find` misses it while `find --archive`, `ls --archive`, and `map --archive` expose it. These are behavioral contracts, not just snapshots of helper return values.

Tests still have boundaries. They do not decide whether a course's actualized summary is honest or whether a lesson teaches well. They do not replace a final Quarto render and visual review. The command tests cover representative options rather than every combination; the focused tests carry much of the locator and error-path detail. Jupyter execution tests also need permission to bind local loopback ports.

For a change to this stack, use `make lint`, `make typecheck`, and `make test` for Python behavior; `make knowledge` for catalog and generated-context consistency; and a site render for the learner-facing output. If a command or option changes, update both `README.md` and `AGENTS.md` so the documented workflow remains executable.


## Working rule

The catalog identifies active work. Course YAML gives a concise, shared contract and keeps intended work distinct from completed work. The home orients learners, the optional overview explains the whole-course technical design, and chapters teach local material. `wt context` points an agent through those layers without loading every sibling chapter; `wt cat --context` handles nearby cells inside one notebook.

The tests protect the mechanical boundaries: registration, discovery, context, cell mutation, execution, archive separation, and publication selection. They make the repository structure dependable while leaving judgments about teaching quality and truthful progress to the author.
